# Computer Vision Lab: Moments, Gradients, Histograms, HOG, and Recognition

## Lab Title

**Feature Extraction for Object Localization, Shape Analysis, and Recognition**

## Topics Covered

### Raw Moments
- Raw Moments and Centroid of a Binary Shape
- Raw Moments of a Small Grayscale Patch
- Raw Moments and Weighted Centroid of a Grayscale Patch
- Effect of Translation and Intensity Scaling on Raw Moments
- Scenario-Based: Conveyor Belt Object Localization

### Central Moments
- Central Moments of a Grayscale Object
- Central Moments of a Binary Object
- Central Moments and Shape Orientation
- Translation Invariance of Central Moments
- Translation Effect on Raw vs Central Moments
- Scenario-Based: Cell Shape Analysis in Microscopy

### Normalized Central Moments
- Compute Normalized Central Moments
- Compute Normalized Central Moments up to Order 3
- Scenario-Based: Logo Recognition Under Scale Change
- Scale Invariance Using Normalized Central Moments
- Shape-Based Recognition Pipeline
- Feature Vector Design from Moments

### Image Gradients
- Gradient vector
- Gradient along x-axis
- Gradient along y-axis
- Gradient in both x and y
- Gradient using forward, backward, and central difference filters
- Sobel filter

### Histogram
- Histogram of binary image
- Histogram of grayscale image
- Normalized histogram of grayscale image
- Limitation: histograms ignore spatial arrangement

### HOG
- Concept of HOG
- Manual HOG understanding
- HOG Descriptor using scikit-image
- Recognition / Detection

## Learning Outcomes

After completing this lab, students should be able to:

1. Compute raw moments of binary and grayscale images.
2. Find centroid and weighted centroid of objects.
3. Analyze translation and intensity scaling effects on raw moments.
4. Compute central moments and explain translation invariance.
5. Estimate object orientation using second-order central moments.
6. Compute normalized central moments and use them for scale-invariant shape description.
7. Design moment-based feature vectors for recognition.
8. Compute image gradients using basic derivative filters and Sobel filters.
9. Compute and interpret binary, grayscale, and normalized histograms.
10. Implement HOG descriptors and use them in a basic recognition pipeline.

## Required Python Libraries

Run the following command in your environment if required:

```bash
pip install numpy matplotlib opencv-python scikit-image scikit-learn
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

from skimage.feature import hog
from skimage import exposure
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

np.random.seed(42)

In [ ]:
def show_image(img, title="Image", cmap="gray", figsize=(5, 5)):
    plt.figure(figsize=figsize)
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.axis("off")
    plt.show()

# Part A: Raw Moments

## Concept

For an image intensity function $I(x,y)$, the raw moment of order $(p,q)$ is:

$$
m_{pq} = \sum_x \sum_y x^p y^q I(x,y)
$$

For a binary image:

$$
I(x,y)=
\begin{cases}
1, & \text{object pixel}\\
0, & \text{background pixel}
\end{cases}
$$

Important raw moments:

$$
m_{00} = \text{area or total intensity}
$$

$$
m_{10} = \sum_x \sum_y xI(x,y)
$$

$$
m_{01} = \sum_x \sum_y yI(x,y)
$$

The centroid is:

$$
\bar{x} = \frac{m_{10}}{m_{00}}, \qquad
\bar{y} = \frac{m_{01}}{m_{00}}
$$

## Task 1: Raw Moments and Centroid of a Binary Shape

In [ ]:
img_binary_rect = np.zeros((100, 100), dtype=np.uint8)

# Draw white rectangle
img_binary_rect[30:70, 40:80] = 1

show_image(img_binary_rect, "Binary Rectangle")

In [ ]:
def raw_moment(img, p, q):
    h, w = img.shape
    moment = 0.0

    for y in range(h):
        for x in range(w):
            moment += (x ** p) * (y ** q) * img[y, x]

    return moment

In [ ]:
m00 = raw_moment(img_binary_rect, 0, 0)
m10 = raw_moment(img_binary_rect, 1, 0)
m01 = raw_moment(img_binary_rect, 0, 1)

cx = m10 / m00
cy = m01 / m00

print("m00 =", m00)
print("m10 =", m10)
print("m01 =", m01)
print("Centroid =", (cx, cy))

In [ ]:
plt.figure(figsize=(5, 5))
plt.imshow(img_binary_rect, cmap="gray")
plt.scatter(cx, cy, c="red", s=80)
plt.title("Binary Shape Centroid")
plt.axis("off")
plt.show()

## Practice Task 1

Create the following binary shapes and compute their centroids:

1. Rectangle  
2. Circle  
3. Triangle  
4. L-shaped object  
5. Two disconnected objects  

For each shape, fill the following table in your lab report:

| Shape | $m_{00}$ | $m_{10}$ | $m_{01}$ | Centroid |
|---|---:|---:|---:|---|
| Rectangle |  |  |  |  |
| Circle |  |  |  |  |
| Triangle |  |  |  |  |
| L-shape |  |  |  |  |
| Two objects |  |  |  |  |

In [ ]:
# Practice helper: generate different binary shapes

def create_binary_shape(shape_type, size=100):
    img = np.zeros((size, size), dtype=np.uint8)

    if shape_type == "rectangle":
        cv2.rectangle(img, (30, 35), (75, 70), 1, -1)

    elif shape_type == "circle":
        cv2.circle(img, (50, 50), 22, 1, -1)

    elif shape_type == "triangle":
        pts = np.array([[50, 20], [20, 80], [80, 80]], dtype=np.int32)
        cv2.fillPoly(img, [pts], 1)

    elif shape_type == "l_shape":
        cv2.rectangle(img, (25, 25), (45, 80), 1, -1)
        cv2.rectangle(img, (25, 60), (80, 80), 1, -1)

    elif shape_type == "two_objects":
        cv2.circle(img, (30, 35), 12, 1, -1)
        cv2.rectangle(img, (60, 60), (85, 85), 1, -1)

    return img


practice_shapes = ["rectangle", "circle", "triangle", "l_shape", "two_objects"]

for shape_name in practice_shapes:
    shape_img = create_binary_shape(shape_name)
    m00 = raw_moment(shape_img, 0, 0)
    m10 = raw_moment(shape_img, 1, 0)
    m01 = raw_moment(shape_img, 0, 1)
    cx = m10 / m00
    cy = m01 / m00

    print(f"\nShape: {shape_name}")
    print("m00 =", m00)
    print("m10 =", m10)
    print("m01 =", m01)
    print("Centroid =", (cx, cy))

    plt.figure(figsize=(4, 4))
    plt.imshow(shape_img, cmap="gray")
    plt.scatter(cx, cy, c="red", s=60)
    plt.title(shape_name)
    plt.axis("off")
    plt.show()

## Task 2: Raw Moments of a Small Grayscale Patch

In [ ]:
patch = np.array([
    [1, 2, 1],
    [0, 4, 2],
    [1, 3, 5]
], dtype=np.float32)

show_image(patch, "Small Grayscale Patch", figsize=(3, 3))
print(patch)

In [ ]:
m00 = raw_moment(patch, 0, 0)
m10 = raw_moment(patch, 1, 0)
m01 = raw_moment(patch, 0, 1)
m20 = raw_moment(patch, 2, 0)
m02 = raw_moment(patch, 0, 2)
m11 = raw_moment(patch, 1, 1)

print("m00 =", m00)
print("m10 =", m10)
print("m01 =", m01)
print("m20 =", m20)
print("m02 =", m02)
print("m11 =", m11)

cx = m10 / m00
cy = m01 / m00

print("Weighted centroid =", (cx, cy))

### Required Explanation

Answer the following in your lab report:

1. Why do bright pixels contribute more to the centroid?
2. Why may the weighted centroid not coincide with the geometric center?
3. What happens if all intensities are multiplied by a constant?

## Task 3: Raw Moments and Weighted Centroid of a Grayscale Patch with Coordinate Offset

Assume the top-left coordinate of the patch is $(5,4)$, not $(0,0)$.

In [ ]:
def raw_moment_with_offset(img, p, q, x0=0, y0=0):
    h, w = img.shape
    moment = 0.0

    for y in range(h):
        for x in range(w):
            actual_x = x + x0
            actual_y = y + y0
            moment += (actual_x ** p) * (actual_y ** q) * img[y, x]

    return moment

In [ ]:
x0, y0 = 5, 4

m00_offset = raw_moment_with_offset(patch, 0, 0, x0, y0)
m10_offset = raw_moment_with_offset(patch, 1, 0, x0, y0)
m01_offset = raw_moment_with_offset(patch, 0, 1, x0, y0)

cx_offset = m10_offset / m00_offset
cy_offset = m01_offset / m00_offset

print("m00 =", m00_offset)
print("m10 =", m10_offset)
print("m01 =", m01_offset)
print("Weighted centroid =", (cx_offset, cy_offset))

## Task 4: Effect of Translation and Intensity Scaling on Raw Moments

Apply:

- translation: $x' = x + 2$
- translation: $y' = y + 1$
- intensity scaling: $I' = 3I$

For raw moments:

$$
m'_{00} = k m_{00}
$$

$$
m'_{10} = k(m_{10} + a m_{00})
$$

$$
m'_{01} = k(m_{01} + b m_{00})
$$

$$
m'_{20} = k(m_{20} + 2am_{10} + a^2m_{00})
$$

$$
m'_{02} = k(m_{02} + 2bm_{01} + b^2m_{00})
$$

$$
m'_{11} = k(m_{11} + am_{01} + bm_{10} + abm_{00})
$$

In [ ]:
def transformed_raw_moments(m00, m10, m01, m20, m02, m11, a, b, k):
    m00_new = k * m00
    m10_new = k * (m10 + a * m00)
    m01_new = k * (m01 + b * m00)
    m20_new = k * (m20 + 2 * a * m10 + (a ** 2) * m00)
    m02_new = k * (m02 + 2 * b * m01 + (b ** 2) * m00)
    m11_new = k * (m11 + a * m01 + b * m10 + a * b * m00)

    return m00_new, m10_new, m01_new, m20_new, m02_new, m11_new

In [ ]:
a = 2
b = 1
k = 3

m00 = raw_moment(patch, 0, 0)
m10 = raw_moment(patch, 1, 0)
m01 = raw_moment(patch, 0, 1)
m20 = raw_moment(patch, 2, 0)
m02 = raw_moment(patch, 0, 2)
m11 = raw_moment(patch, 1, 1)

new_moments = transformed_raw_moments(m00, m10, m01, m20, m02, m11, a, b, k)

print("Original moments:")
print("m00, m10, m01, m20, m02, m11 =", (m00, m10, m01, m20, m02, m11))

print("\nTransformed raw moments:")
print("m00', m10', m01', m20', m02', m11' =", new_moments)

new_m00, new_m10, new_m01, *_ = new_moments
new_cx = new_m10 / new_m00
new_cy = new_m01 / new_m00

print("\nNew weighted centroid =", (new_cx, new_cy))

### Required Questions

1. Does intensity scaling change $m_{00}$?
2. Does intensity scaling change the centroid?
3. Does translation change the centroid?
4. Are raw moments translation invariant?
5. Why are raw moments useful but limited for shape recognition?

## Scenario Task: Conveyor Belt Object Localization

A factory conveyor belt camera captures binary images of objects. The system must find the object location so that a robotic arm can pick it.

### Student Task

1. Create or load a binary image containing one object.
2. Compute:
   - $m_{00}$
   - $m_{10}$
   - $m_{01}$
   - centroid
3. Draw the centroid on the image.
4. Use the centroid as the estimated picking point.
5. Test the system after translating the object.

In [ ]:
belt_img = np.zeros((160, 240), dtype=np.uint8)

# Simulated object on conveyor belt
cv2.ellipse(belt_img, (130, 80), (40, 20), 25, 0, 360, 1, -1)

m00 = raw_moment(belt_img, 0, 0)
m10 = raw_moment(belt_img, 1, 0)
m01 = raw_moment(belt_img, 0, 1)

cx = m10 / m00
cy = m01 / m00

print("Object area:", m00)
print("Object centroid:", (cx, cy))
print("Robot picking point:", (cx, cy))

plt.figure(figsize=(7, 4))
plt.imshow(belt_img, cmap="gray")
plt.scatter(cx, cy, c="red", s=80)
plt.title("Conveyor Belt Object Localization")
plt.axis("off")
plt.show()

In [ ]:
# Advanced extension: centroid of each connected component

multi_obj = np.zeros((160, 240), dtype=np.uint8)
cv2.circle(multi_obj, (60, 70), 20, 1, -1)
cv2.rectangle(multi_obj, (140, 50), (190, 100), 1, -1)

num_labels, labels = cv2.connectedComponents(multi_obj.astype(np.uint8))

plt.figure(figsize=(7, 4))
plt.imshow(multi_obj, cmap="gray")

for label_id in range(1, num_labels):
    component = (labels == label_id).astype(np.uint8)

    m00 = raw_moment(component, 0, 0)
    m10 = raw_moment(component, 1, 0)
    m01 = raw_moment(component, 0, 1)

    cx = m10 / m00
    cy = m01 / m00

    print(f"Component {label_id}: area={m00}, centroid=({cx:.2f}, {cy:.2f})")
    plt.scatter(cx, cy, s=80)

plt.title("Connected Component Centroids")
plt.axis("off")
plt.show()

# Part B: Central Moments

## Concept

Central moments are computed relative to the centroid:

$$
\mu_{pq} = \sum_x \sum_y (x - \bar{x})^p (y - \bar{y})^q I(x,y)
$$

Important properties:

- Central moments are translation invariant.
- Second-order central moments describe spread and orientation.
- Third-order central moments describe asymmetry.

## Task 5: Central Moments of a Binary Object

In [ ]:
def centroid(img):
    m00 = raw_moment(img, 0, 0)

    if m00 == 0:
        raise ValueError("Cannot compute centroid because m00 is zero.")

    m10 = raw_moment(img, 1, 0)
    m01 = raw_moment(img, 0, 1)

    cx = m10 / m00
    cy = m01 / m00

    return cx, cy

In [ ]:
def central_moment(img, p, q):
    h, w = img.shape
    cx, cy = centroid(img)

    mu = 0.0

    for y in range(h):
        for x in range(w):
            mu += ((x - cx) ** p) * ((y - cy) ** q) * img[y, x]

    return mu

In [ ]:
mu20 = central_moment(img_binary_rect, 2, 0)
mu02 = central_moment(img_binary_rect, 0, 2)
mu11 = central_moment(img_binary_rect, 1, 1)

print("mu20 =", mu20)
print("mu02 =", mu02)
print("mu11 =", mu11)

## Task 6: Central Moments of a Grayscale Object

In [ ]:
gray_obj = np.zeros((100, 100), dtype=np.float32)

cv2.circle(gray_obj, (50, 50), 20, 100, -1)
cv2.circle(gray_obj, (60, 45), 10, 200, -1)

show_image(gray_obj, "Grayscale Object")

cx, cy = centroid(gray_obj)

mu20 = central_moment(gray_obj, 2, 0)
mu02 = central_moment(gray_obj, 0, 2)
mu11 = central_moment(gray_obj, 1, 1)

print("Weighted centroid =", (cx, cy))
print("mu20 =", mu20)
print("mu02 =", mu02)
print("mu11 =", mu11)

### Required Explanation

Explain why the central moments of a grayscale object are affected by the intensity distribution.

## Task 7: Central Moments and Shape Orientation

Object orientation can be estimated using:

$$
\theta = \frac{1}{2}\tan^{-1}\left(\frac{2\mu_{11}}{\mu_{20}-\mu_{02}}\right)
$$

In [ ]:
def shape_orientation(img):
    mu20 = central_moment(img, 2, 0)
    mu02 = central_moment(img, 0, 2)
    mu11 = central_moment(img, 1, 1)

    theta = 0.5 * np.arctan2(2 * mu11, mu20 - mu02)

    return theta

In [ ]:
theta = shape_orientation(img_binary_rect)

print("Orientation in radians =", theta)
print("Orientation in degrees =", np.degrees(theta))

### Advanced Task: Rotated Ellipse Orientation

In [ ]:
ellipse_img = np.zeros((150, 150), dtype=np.uint8)

cv2.ellipse(
    ellipse_img,
    center=(75, 75),
    axes=(40, 15),
    angle=30,
    startAngle=0,
    endAngle=360,
    color=1,
    thickness=-1
)

show_image(ellipse_img, "Rotated Ellipse")

theta = shape_orientation(ellipse_img)
print("Estimated orientation =", np.degrees(theta))

## Task 8: Translation Invariance of Central Moments

In [ ]:
shape1 = np.zeros((120, 120), dtype=np.uint8)
cv2.rectangle(shape1, (30, 40), (70, 80), 1, -1)

shape2 = np.zeros((120, 120), dtype=np.uint8)
cv2.rectangle(shape2, (50, 60), (90, 100), 1, -1)

show_image(shape1, "Original Shape")
show_image(shape2, "Translated Shape")

In [ ]:
for name, shape in [("Original", shape1), ("Translated", shape2)]:
    print("\n", name)
    print("Centroid =", centroid(shape))
    print("Raw m10 =", raw_moment(shape, 1, 0))
    print("Raw m01 =", raw_moment(shape, 0, 1))
    print("Central mu20 =", central_moment(shape, 2, 0))
    print("Central mu02 =", central_moment(shape, 0, 2))
    print("Central mu11 =", central_moment(shape, 1, 1))

### Required Observation

| Moment Type | Effect of Translation |
|---|---|
| Raw moments | Change |
| Central moments | Mostly unchanged |
| Centroid | Moves with object |
| Shape orientation | Should remain same |

## Scenario Task: Cell Shape Analysis in Microscopy

Microscopy images often contain cells with different shapes. Shape analysis can help classify cells as:

- round,
- elongated,
- irregular.

In [ ]:
cell_round = np.zeros((150, 150), dtype=np.uint8)
cv2.circle(cell_round, (75, 75), 30, 1, -1)

cell_elongated = np.zeros((150, 150), dtype=np.uint8)
cv2.ellipse(cell_elongated, (75, 75), (45, 15), 25, 0, 360, 1, -1)

cell_irregular = np.zeros((150, 150), dtype=np.uint8)
pts = np.array([[60, 30], [100, 45], [120, 90], [80, 120], [45, 100], [40, 55]], dtype=np.int32)
cv2.fillPoly(cell_irregular, [pts], 1)

cells = {
    "Round Cell": cell_round,
    "Elongated Cell": cell_elongated,
    "Irregular Cell": cell_irregular
}

for name, cell in cells.items():
    show_image(cell, name)

In [ ]:
def elongation_measure(img):
    mu20 = central_moment(img, 2, 0)
    mu02 = central_moment(img, 0, 2)

    denominator = min(mu20, mu02)
    if denominator == 0:
        return np.inf

    return max(mu20, mu02) / denominator

In [ ]:
for name, cell in cells.items():
    theta = shape_orientation(cell)
    elongation = elongation_measure(cell)

    print("\n", name)
    print("Centroid =", centroid(cell))
    print("mu20 =", central_moment(cell, 2, 0))
    print("mu02 =", central_moment(cell, 0, 2))
    print("mu11 =", central_moment(cell, 1, 1))
    print("Orientation =", np.degrees(theta))
    print("Elongation measure =", elongation)

# Part C: Normalized Central Moments

## Concept

Normalized central moments are used to reduce the effect of scale:

$$
\eta_{pq} = \frac{\mu_{pq}}{\mu_{00}^{\gamma}}
$$

where:

$$
\gamma = 1 + \frac{p+q}{2}
$$

Thus:

$$
\eta_{pq} = \frac{\mu_{pq}}{\mu_{00}^{1+\frac{p+q}{2}}}
$$

Important notes:

- Central moments are translation invariant.
- Normalized central moments are scale invariant for geometric scaling.
- Normalized central moments are useful for shape-based recognition.
- For grayscale images, pure intensity scaling can still affect normalized central moments unless intensity normalization is handled separately.

## Task 9: Compute Normalized Central Moments

In [ ]:
def normalized_central_moment(img, p, q):
    mu_pq = central_moment(img, p, q)
    mu00 = central_moment(img, 0, 0)

    if mu00 == 0:
        raise ValueError("Cannot compute normalized central moment because mu00 is zero.")

    gamma = 1 + ((p + q) / 2)
    eta = mu_pq / (mu00 ** gamma)

    return eta

In [ ]:
eta20 = normalized_central_moment(img_binary_rect, 2, 0)
eta02 = normalized_central_moment(img_binary_rect, 0, 2)
eta11 = normalized_central_moment(img_binary_rect, 1, 1)

print("eta20 =", eta20)
print("eta02 =", eta02)
print("eta11 =", eta11)

## Task 10: Compute Normalized Central Moments up to Order 3

In [ ]:
orders = [
    (2, 0),
    (0, 2),
    (1, 1),
    (3, 0),
    (0, 3),
    (2, 1),
    (1, 2)
]

for p, q in orders:
    eta = normalized_central_moment(img_binary_rect, p, q)
    print(f"eta{p}{q} = {eta}")

## Task 11: Scale Invariance Using Normalized Central Moments

In [ ]:
shape_small = np.zeros((120, 120), dtype=np.uint8)
cv2.circle(shape_small, (60, 60), 15, 1, -1)

shape_large = np.zeros((120, 120), dtype=np.uint8)
cv2.circle(shape_large, (60, 60), 30, 1, -1)

show_image(shape_small, "Small Circle")
show_image(shape_large, "Large Circle")

In [ ]:
for name, shape in [("Small Circle", shape_small), ("Large Circle", shape_large)]:
    print("\n", name)

    print("m00 =", raw_moment(shape, 0, 0))
    print("mu20 =", central_moment(shape, 2, 0))
    print("mu02 =", central_moment(shape, 0, 2))
    print("eta20 =", normalized_central_moment(shape, 2, 0))
    print("eta02 =", normalized_central_moment(shape, 0, 2))

### Expected Observation

| Feature | Small Shape vs Large Shape |
|---|---|
| $m_{00}$ | Changes significantly |
| $\mu_{20}$, $\mu_{02}$ | Change significantly |
| $\eta_{20}$, $\eta_{02}$ | More stable |
| Centroid | May remain same if centered |

## Scenario Task: Logo Recognition Under Scale Change

A logo may appear at different sizes in an image. The recognition system should identify it as the same logo.

In [ ]:
def moment_feature_vector(img):
    features = []

    orders = [
        (2, 0),
        (0, 2),
        (1, 1),
        (3, 0),
        (0, 3),
        (2, 1),
        (1, 2)
    ]

    for p, q in orders:
        features.append(normalized_central_moment(img, p, q))

    return np.array(features, dtype=np.float64)

In [ ]:
f_small = moment_feature_vector(shape_small)
f_large = moment_feature_vector(shape_large)

distance = np.linalg.norm(f_small - f_large)

print("Feature vector small:", f_small)
print("Feature vector large:", f_large)
print("Euclidean distance:", distance)

### Required Interpretation

Explain:

1. Why normalized moments are better than raw moments for scale-changing objects.
2. Why moments alone may not be enough for complex object recognition.
3. Why third-order moments help describe asymmetry.

## Scenario-Based Design Task: Shape-Based Recognition Pipeline

Required pipeline:

```text
Input Image
     ↓
Grayscale Conversion
     ↓
Thresholding / Segmentation
     ↓
Noise Removal
     ↓
Connected Component Extraction
     ↓
Moment Feature Extraction
     ↓
Feature Vector Construction
     ↓
Classifier / Rule-Based Decision
     ↓
Recognition Result
```

Required features should include at least five from:

$$
m_{00}, m_{10}, m_{01}, \mu_{20}, \mu_{02}, \mu_{11}, \eta_{20}, \eta_{02}, \eta_{11}, \eta_{30}, \eta_{03}, \eta_{21}, \eta_{12}
$$

In [ ]:
def extract_moment_feature_dictionary(img):
    binary = (img > 0).astype(np.uint8)

    features = {
        "m00": raw_moment(binary, 0, 0),
        "m10": raw_moment(binary, 1, 0),
        "m01": raw_moment(binary, 0, 1),
        "mu20": central_moment(binary, 2, 0),
        "mu02": central_moment(binary, 0, 2),
        "mu11": central_moment(binary, 1, 1),
        "eta20": normalized_central_moment(binary, 2, 0),
        "eta02": normalized_central_moment(binary, 0, 2),
        "eta11": normalized_central_moment(binary, 1, 1),
        "eta30": normalized_central_moment(binary, 3, 0),
        "eta03": normalized_central_moment(binary, 0, 3),
        "eta21": normalized_central_moment(binary, 2, 1),
        "eta12": normalized_central_moment(binary, 1, 2),
    }

    return features


features_dict = extract_moment_feature_dictionary(shape_small)

for key, value in features_dict.items():
    print(key, "=", value)

# Part D: Image Gradients

## Concept

The image gradient measures intensity change.

The gradient vector is:

$$
\nabla I = \left[\frac{\partial I}{\partial x}, \frac{\partial I}{\partial y}\right]
$$

where:

- $G_x$ measures change along x-axis,
- $G_y$ measures change along y-axis.

Gradient magnitude:

$$
G = \sqrt{G_x^2 + G_y^2}
$$

Gradient direction:

$$
\theta = \tan^{-1}\left(\frac{G_y}{G_x}\right)
$$

Gradients are essential for:

- edge detection,
- corner detection,
- texture analysis,
- HOG descriptors.

## Task 12: Gradient Along X-Axis

In [ ]:
grad_img = np.zeros((100, 100), dtype=np.float32)
grad_img[:, 50:] = 255

show_image(grad_img, "Vertical Edge Image")

In [ ]:
Gx = np.zeros_like(grad_img)
Gx[:, :-1] = grad_img[:, 1:] - grad_img[:, :-1]

show_image(Gx, "Gradient Along X-axis")

## Task 13: Gradient Along Y-Axis

In [ ]:
img_y = np.zeros((100, 100), dtype=np.float32)
img_y[50:, :] = 255

show_image(img_y, "Horizontal Edge Image")

In [ ]:
Gy = np.zeros_like(img_y)
Gy[:-1, :] = img_y[1:, :] - img_y[:-1, :]

show_image(Gy, "Gradient Along Y-axis")

## Task 14: Gradient in Both Directions

In [ ]:
img_edges = np.zeros((100, 100), dtype=np.float32)
img_edges[30:70, 30:70] = 255

show_image(img_edges, "Square Image")

In [ ]:
Gx = np.zeros_like(img_edges)
Gy = np.zeros_like(img_edges)

Gx[:, :-1] = img_edges[:, 1:] - img_edges[:, :-1]
Gy[:-1, :] = img_edges[1:, :] - img_edges[:-1, :]

G_mag = np.sqrt(Gx ** 2 + Gy ** 2)
G_dir = np.arctan2(Gy, Gx)

show_image(Gx, "Gx")
show_image(Gy, "Gy")
show_image(G_mag, "Gradient Magnitude")

## Task 15: Forward, Backward, and Central Difference Filters

Forward difference:

$$
G_x(x,y) = I(x+1,y) - I(x,y)
$$

Backward difference:

$$
G_x(x,y) = I(x,y) - I(x-1,y)
$$

Central difference:

$$
G_x(x,y) = \frac{I(x+1,y) - I(x-1,y)}{2}
$$

In [ ]:
def forward_difference_x(img):
    gx = np.zeros_like(img, dtype=np.float32)
    gx[:, :-1] = img[:, 1:] - img[:, :-1]
    return gx


def backward_difference_x(img):
    gx = np.zeros_like(img, dtype=np.float32)
    gx[:, 1:] = img[:, 1:] - img[:, :-1]
    return gx


def central_difference_x(img):
    gx = np.zeros_like(img, dtype=np.float32)
    gx[:, 1:-1] = (img[:, 2:] - img[:, :-2]) / 2
    return gx

In [ ]:
gx_forward = forward_difference_x(img_edges)
gx_backward = backward_difference_x(img_edges)
gx_central = central_difference_x(img_edges)

show_image(gx_forward, "Forward Difference X")
show_image(gx_backward, "Backward Difference X")
show_image(gx_central, "Central Difference X")

### Student Comparison Task

Apply all three filters to the same image and compare:

| Method | Edge Thickness | Noise Sensitivity | Accuracy |
|---|---|---|---|
| Forward Difference |  |  |  |
| Backward Difference |  |  |  |
| Central Difference |  |  |  |

## Task 16: Sobel Filter

Sobel filters combine differentiation and smoothing.

$$
S_x =
\begin{bmatrix}
-1 & 0 & 1\\
-2 & 0 & 2\\
-1 & 0 & 1
\end{bmatrix}
$$

$$
S_y =
\begin{bmatrix}
-1 & -2 & -1\\
0 & 0 & 0\\
1 & 2 & 1
\end{bmatrix}
$$

In [ ]:
sobel_x = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=np.float32)

sobel_y = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
], dtype=np.float32)

Gx_sobel = cv2.filter2D(img_edges, cv2.CV_32F, sobel_x)
Gy_sobel = cv2.filter2D(img_edges, cv2.CV_32F, sobel_y)

Gmag_sobel = np.sqrt(Gx_sobel ** 2 + Gy_sobel ** 2)

show_image(Gx_sobel, "Sobel Gx")
show_image(Gy_sobel, "Sobel Gy")
show_image(Gmag_sobel, "Sobel Gradient Magnitude")

### Advanced Task: Gradient Stability Under Noise

Add Gaussian noise and compare:

1. Forward difference gradient.
2. Central difference gradient.
3. Sobel gradient.

In [ ]:
noise = np.random.normal(0, 25, img_edges.shape)
noisy_edges = np.clip(img_edges + noise, 0, 255).astype(np.float32)

gx_forward_noisy = forward_difference_x(noisy_edges)
gx_central_noisy = central_difference_x(noisy_edges)
gx_sobel_noisy = cv2.filter2D(noisy_edges, cv2.CV_32F, sobel_x)

show_image(noisy_edges, "Noisy Image")
show_image(np.abs(gx_forward_noisy), "Forward Difference on Noisy Image")
show_image(np.abs(gx_central_noisy), "Central Difference on Noisy Image")
show_image(np.abs(gx_sobel_noisy), "Sobel X on Noisy Image")

# Part E: Histograms

## Concept

A histogram counts how many pixels have each intensity value.

For an 8-bit grayscale image:

$$
I(x,y) \in [0,255]
$$

A full grayscale histogram has 256 bins.

Important limitation:

> Histograms describe intensity distribution but ignore spatial arrangement.

This means two visually different images can have the same or very similar histograms.

## Task 17: Histogram of Binary Image

In [ ]:
binary_img = np.zeros((100, 100), dtype=np.uint8)
binary_img[30:70, 30:70] = 255

hist_binary = cv2.calcHist([binary_img], [0], None, [256], [0, 256])

show_image(binary_img, "Binary Image")

plt.figure()
plt.plot(hist_binary)
plt.title("Binary Image Histogram")
plt.xlabel("Intensity")
plt.ylabel("Pixel Count")
plt.show()

### Required Observation

Students should observe two dominant bins:

- intensity 0
- intensity 255

## Task 18: Histogram of Grayscale Image

In [ ]:
gray_img = np.tile(np.arange(0, 256, dtype=np.uint8), (100, 1))

show_image(gray_img, "Grayscale Gradient Image")

hist_gray = cv2.calcHist([gray_img], [0], None, [256], [0, 256])

plt.figure()
plt.plot(hist_gray)
plt.title("Grayscale Histogram")
plt.xlabel("Intensity")
plt.ylabel("Pixel Count")
plt.show()

## Task 19: Normalized Histogram

In [ ]:
hist = cv2.calcHist([gray_img], [0], None, [256], [0, 256])
hist_norm = hist / hist.sum()

plt.figure()
plt.plot(hist_norm)
plt.title("Normalized Grayscale Histogram")
plt.xlabel("Intensity")
plt.ylabel("Probability")
plt.show()

print("Sum of normalized histogram =", hist_norm.sum())

## Task 20: Histograms Ignore Spatial Arrangement

In [ ]:
img_a = np.zeros((100, 100), dtype=np.uint8)
img_a[:, :50] = 255

img_b = np.zeros((100, 100), dtype=np.uint8)
img_b[::2, :] = 255

show_image(img_a, "Image A")
show_image(img_b, "Image B")

In [ ]:
hist_a = cv2.calcHist([img_a], [0], None, [256], [0, 256])
hist_b = cv2.calcHist([img_b], [0], None, [256], [0, 256])

plt.figure()
plt.plot(hist_a, label="Image A")
plt.plot(hist_b, label="Image B", linestyle="--")
plt.legend()
plt.title("Histogram Comparison")
plt.xlabel("Intensity")
plt.ylabel("Pixel Count")
plt.show()

### Required Explanation

Explain:

1. Why both images may have similar histograms.
2. Why their visual structure is different.
3. Why histogram alone is not sufficient for object recognition.

# Part F: HOG — Histogram of Oriented Gradients

## Concept

HOG describes an image using local gradient directions.

Main pipeline:

```text
Image
  ↓
Compute Gradients
  ↓
Compute Gradient Magnitude and Orientation
  ↓
Divide Image into Cells
  ↓
Build Orientation Histogram per Cell
  ↓
Normalize Over Blocks
  ↓
Generate HOG Feature Descriptor
  ↓
Use for Recognition / Detection
```

HOG is useful because:

- it captures shape and edge structure,
- it is more spatially informative than a normal histogram,
- it is effective for object detection.

## Task 21: Manual Understanding of HOG

In [ ]:
hog_input = np.zeros((64, 64), dtype=np.float32)
cv2.rectangle(hog_input, (20, 15), (45, 50), 255, -1)

Gx = cv2.Sobel(hog_input, cv2.CV_32F, 1, 0, ksize=3)
Gy = cv2.Sobel(hog_input, cv2.CV_32F, 0, 1, ksize=3)

magnitude = np.sqrt(Gx ** 2 + Gy ** 2)
orientation = np.degrees(np.arctan2(Gy, Gx)) % 180

show_image(hog_input, "Input Shape")
show_image(magnitude, "Gradient Magnitude")
show_image(orientation, "Gradient Orientation")

### Create Orientation Histogram for One Cell

In [ ]:
cell_mag = magnitude[0:8, 0:8]
cell_ori = orientation[0:8, 0:8]

bins = 9
hist = np.zeros(bins)

bin_width = 180 / bins

for i in range(cell_mag.shape[0]):
    for j in range(cell_mag.shape[1]):
        angle = cell_ori[i, j]
        mag = cell_mag[i, j]

        bin_index = int(angle // bin_width)

        if bin_index == bins:
            bin_index = bins - 1

        hist[bin_index] += mag

print("HOG histogram for one cell:")
print(hist)

### Required Observation

Explain:

1. Why strong edges contribute more to the histogram.
2. Why gradient magnitude is used as a weight.
3. Why orientations are usually grouped into bins.

## Task 22: HOG Descriptor Using scikit-image

In [ ]:
hog_img = np.zeros((128, 64), dtype=np.uint8)

cv2.rectangle(hog_img, (20, 20), (45, 100), 255, -1)

features, hog_image = hog(
    hog_img,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    block_norm="L2-Hys",
    visualize=True,
    feature_vector=True
)

hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))

show_image(hog_img, "Input Image")
show_image(hog_image_rescaled, "HOG Visualization")

print("HOG feature vector length:", len(features))

### Required Questions

1. What does `orientations=9` mean?
2. What does `pixels_per_cell=(8,8)` mean?
3. Why are blocks normalized?
4. Why is HOG better than a simple grayscale histogram for object detection?

## Task 23: HOG-Based Recognition

Create synthetic shapes and classify them using HOG descriptors.

In [ ]:
def create_shape(shape_type, size=64):
    img = np.zeros((size, size), dtype=np.uint8)

    if shape_type == "vertical_rectangle":
        cv2.rectangle(img, (25, 10), (39, 54), 255, -1)

    elif shape_type == "horizontal_rectangle":
        cv2.rectangle(img, (10, 25), (54, 39), 255, -1)

    elif shape_type == "circle":
        cv2.circle(img, (32, 32), 18, 255, -1)

    elif shape_type == "triangle":
        pts = np.array([[32, 10], [10, 54], [54, 54]], np.int32)
        cv2.fillPoly(img, [pts], 255)

    else:
        raise ValueError("Unknown shape type.")

    return img


def translate_image(img, tx, ty):
    h, w = img.shape
    M = np.float32([[1, 0, tx], [0, 1, ty]])
    shifted = cv2.warpAffine(img, M, (w, h), borderValue=0)
    return shifted

In [ ]:
shape_classes = [
    "vertical_rectangle",
    "horizontal_rectangle",
    "circle",
    "triangle"
]

for shape_type in shape_classes:
    sample = create_shape(shape_type)
    show_image(sample, shape_type, figsize=(3, 3))

In [ ]:
X = []
y = []

for label, shape_type in enumerate(shape_classes):
    base_img = create_shape(shape_type)

    for i in range(30):
        tx = np.random.randint(-5, 6)
        ty = np.random.randint(-5, 6)

        sample = translate_image(base_img, tx, ty)

        features = hog(
            sample,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm="L2-Hys",
            feature_vector=True
        )

        X.append(features)
        y.append(label)

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)
print("Labels shape:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

clf = KNeighborsClassifier(n_neighbors=3)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred, target_names=shape_classes))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

### Required Interpretation

Discuss:

1. Which shapes were recognized correctly?
2. Which shapes were confused?
3. Why is HOG useful for shape recognition?
4. What happens if the object is rotated?
5. What happens if the object is scaled?

# Part G: Final Combined Recognition / Detection Pipeline

Students must design a complete recognition system using:

1. Moment features.
2. Histogram features.
3. HOG features.

Required dataset:

```text
Class 1: Circle
Class 2: Triangle
Class 3: Vertical Rectangle
Class 4: Horizontal Rectangle
```

For each class, generate at least:

```text
30 training images
10 testing images
```

Each image should include variation in:

- translation,
- slight scaling,
- slight noise,
- brightness change.

## Feature Extraction Functions

In [ ]:
def extract_histogram_features(img):
    hist = cv2.calcHist([img], [0], None, [32], [0, 256])
    hist = hist.flatten()
    hist = hist / (hist.sum() + 1e-8)
    return hist


def extract_hog_features(img):
    features = hog(
        img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True
    )
    return features


def extract_moment_features(img):
    binary = (img > 127).astype(np.uint8)

    # Avoid zero-object cases
    if binary.sum() == 0:
        return np.zeros(7, dtype=np.float64)

    features = []

    for p, q in [
        (2, 0),
        (0, 2),
        (1, 1),
        (3, 0),
        (0, 3),
        (2, 1),
        (1, 2)
    ]:
        features.append(normalized_central_moment(binary, p, q))

    return np.array(features, dtype=np.float64)


def extract_combined_features(img):
    hist_features = extract_histogram_features(img)
    hog_features = extract_hog_features(img)
    moment_features = extract_moment_features(img)

    combined = np.concatenate([
        hist_features,
        hog_features,
        moment_features
    ])

    return combined

## Data Augmentation Helper Functions

In [ ]:
def scale_shape_image(img, scale_factor):
    h, w = img.shape
    scaled = cv2.resize(img, None, fx=scale_factor, fy=scale_factor, interpolation=cv2.INTER_NEAREST)

    canvas = np.zeros_like(img)

    sh, sw = scaled.shape
    crop_h = min(h, sh)
    crop_w = min(w, sw)

    y_start_canvas = (h - crop_h) // 2
    x_start_canvas = (w - crop_w) // 2

    y_start_scaled = (sh - crop_h) // 2
    x_start_scaled = (sw - crop_w) // 2

    canvas[
        y_start_canvas:y_start_canvas + crop_h,
        x_start_canvas:x_start_canvas + crop_w
    ] = scaled[
        y_start_scaled:y_start_scaled + crop_h,
        x_start_scaled:x_start_scaled + crop_w
    ]

    return canvas


def apply_brightness_change(img, factor):
    changed = img.astype(np.float32) * factor
    changed = np.clip(changed, 0, 255).astype(np.uint8)
    return changed


def add_gaussian_noise(img, mean=0, std=10):
    noise = np.random.normal(mean, std, img.shape)
    noisy = img.astype(np.float32) + noise
    noisy = np.clip(noisy, 0, 255).astype(np.uint8)
    return noisy


def augment_shape_image(base_img):
    scale_factor = np.random.uniform(0.85, 1.15)
    tx = np.random.randint(-5, 6)
    ty = np.random.randint(-5, 6)
    brightness = np.random.uniform(0.8, 1.2)

    img = scale_shape_image(base_img, scale_factor)
    img = translate_image(img, tx, ty)
    img = apply_brightness_change(img, brightness)
    img = add_gaussian_noise(img, mean=0, std=10)

    return img

## Final Classification Task

In [ ]:
X_combined = []
y_combined = []

for label, shape_type in enumerate(shape_classes):
    base_img = create_shape(shape_type)

    for i in range(40):
        sample = augment_shape_image(base_img)
        features = extract_combined_features(sample)

        X_combined.append(features)
        y_combined.append(label)

X_combined = np.array(X_combined)
y_combined = np.array(y_combined)

print("Feature matrix:", X_combined.shape)
print("Label vector:", y_combined.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_combined,
    y_combined,
    test_size=0.3,
    random_state=42,
    stratify=y_combined
)

clf_combined = KNeighborsClassifier(n_neighbors=3)
clf_combined.fit(X_train, y_train)

y_pred = clf_combined.predict(X_test)

print(classification_report(y_test, y_pred, target_names=shape_classes))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## Optional Comparison: Moment Features vs Histogram Features vs HOG Features

Students can compare classifier performance using each feature type individually.

In [ ]:
def build_dataset_with_feature_type(feature_type="hog", samples_per_class=40):
    X_data = []
    y_data = []

    for label, shape_type in enumerate(shape_classes):
        base_img = create_shape(shape_type)

        for i in range(samples_per_class):
            sample = augment_shape_image(base_img)

            if feature_type == "histogram":
                features = extract_histogram_features(sample)
            elif feature_type == "hog":
                features = extract_hog_features(sample)
            elif feature_type == "moment":
                features = extract_moment_features(sample)
            elif feature_type == "combined":
                features = extract_combined_features(sample)
            else:
                raise ValueError("Unsupported feature type.")

            X_data.append(features)
            y_data.append(label)

    return np.array(X_data), np.array(y_data)


for feature_type in ["histogram", "moment", "hog", "combined"]:
    print("\nFeature type:", feature_type)

    X_ft, y_ft = build_dataset_with_feature_type(feature_type, samples_per_class=30)

    X_train, X_test, y_train, y_test = train_test_split(
        X_ft,
        y_ft,
        test_size=0.3,
        random_state=42,
        stratify=y_ft
    )

    clf = KNeighborsClassifier(n_neighbors=3)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)

    print(classification_report(y_test, y_pred, target_names=shape_classes))

# Final Lab Report Requirements

Each student must submit a report with the following sections.

## Section 1: Raw Moments

Include:

- binary shape centroid,
- grayscale weighted centroid,
- effect of translation,
- effect of intensity scaling,
- conveyor belt localization result.

## Section 2: Central Moments

Include:

- central moments of binary object,
- central moments of grayscale object,
- orientation result,
- translation invariance comparison,
- cell shape analysis.

## Section 3: Normalized Central Moments

Include:

- normalized moments up to order 3,
- scale invariance experiment,
- logo recognition under scale change,
- moment-based feature vector.

## Section 4: Gradients

Include:

- $G_x$,
- $G_y$,
- gradient magnitude,
- forward difference,
- backward difference,
- central difference,
- Sobel result.

## Section 5: Histograms

Include:

- binary histogram,
- grayscale histogram,
- normalized histogram,
- explanation of spatial limitation.

## Section 6: HOG and Recognition

Include:

- HOG visualization,
- HOG descriptor length,
- classifier result,
- classification report,
- confusion matrix.

## Section 7: Discussion Questions

Answer:

1. Why are raw moments not translation invariant?
2. Why are central moments translation invariant?
3. Why are normalized central moments useful for scale-invariant recognition?
4. Why does a histogram ignore spatial arrangement?
5. Why is HOG better than simple intensity histogram for shape recognition?
6. Which feature type performed best in recognition?
7. What are the limitations of moment-based recognition?
8. What are the limitations of HOG under rotation and scale changes?

# Marking Rubric

Total: **100 marks**

| Component | Marks |
|---|---:|
| Correct implementation of raw moments | 10 |
| Correct centroid and weighted centroid computation | 10 |
| Translation and intensity scaling analysis | 10 |
| Central moments implementation | 10 |
| Orientation estimation using central moments | 10 |
| Normalized central moments up to order 3 | 10 |
| Gradient implementation and comparison | 10 |
| Histogram computation and interpretation | 10 |
| HOG implementation and visualization | 10 |
| Final recognition pipeline and report quality | 10 |

# Viva / Oral Questions

1. What does $m_{00}$ represent in a binary image?
2. What does $m_{00}$ represent in a grayscale image?
3. Why is the grayscale centroid called a weighted centroid?
4. What happens to the centroid if all intensities are multiplied by 3?
5. Why do raw moments change after translation?
6. Why are central moments translation invariant?
7. What do $\mu_{20}$, $\mu_{02}$, and $\mu_{11}$ describe?
8. How can central moments be used to estimate orientation?
9. Why are normalized central moments useful?
10. What is the difference between a histogram and HOG?
11. Why does HOG use gradient orientations?
12. Why does HOG normalize blocks?
13. Why may HOG fail when objects are heavily rotated?
14. Which features are more suitable for shape recognition: histograms, moments, or HOG? Justify.

# Suggested Submission Format

Students should submit:

```text
1. Python notebook file: CV_Lab_Moments_Gradients_Histograms_HOG.ipynb
2. PDF lab report
3. Output images
4. Final classifier results
5. Short discussion answers
```